[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/05-from-models-to-llms/code/preliminary_concepts_of_LLMs.ipynb)

# Class 3.5: From models to LLMs

The slides explain decoding. Here you first see temperature and top-p on a toy distribution, in exact numbers, then generate text with a local Ollama model and sweep the knobs.

What we will cover:
- temperature: sharpening and flattening a distribution
- top-p (nucleus): keeping the most likely tokens
- sampling a next token
- generating with a local model, sweeping temperature and top-p


## Temperature sharpens or flattens

Divide the scores by `T` before softmax. Low `T` concentrates probability on the top token; high `T` spreads it out.

In [1]:
import numpy as np

tokens = ["mat", "rug", "sofa", "floor", "cliff"]
logits = np.array([3.0, 2.0, 1.0, 0.0, -2.0])

def softmax_T(logits, T):
    z = logits / T
    e = np.exp(z - z.max())
    return e / e.sum()

for T in [0.5, 1.0, 2.0]:
    p = softmax_T(logits, T)
    print(f"T={T}:", {t: round(float(v), 2) for t, v in zip(tokens, p)})

T=0.5: {'mat': 0.86, 'rug': 0.12, 'sofa': 0.02, 'floor': 0.0, 'cliff': 0.0}
T=1.0: {'mat': 0.64, 'rug': 0.24, 'sofa': 0.09, 'floor': 0.03, 'cliff': 0.0}
T=2.0: {'mat': 0.44, 'rug': 0.27, 'sofa': 0.16, 'floor': 0.1, 'cliff': 0.04}


## Top-p (nucleus): keep the smallest set that reaches p

Sort tokens by probability, add them up until the total reaches `p`, and keep only those. The unlikely tail is dropped.

In [2]:
def nucleus(probs, tokens, p=0.9):
    order = np.argsort(probs)[::-1]
    kept, total = [], 0.0
    for i in order:
        kept.append(tokens[i]); total += probs[i]
        if total >= p:
            break
    return kept

p = softmax_T(logits, 1.0)
keep = nucleus(p, tokens, p=0.9)
print("nucleus (p=0.9):", keep)
print("dropped tail    :", [t for t in tokens if t not in keep])

nucleus (p=0.9): ['mat', 'rug', 'sofa']
dropped tail    : ['floor', 'cliff']


## Top-k: keep a fixed number of tokens

Top-k is the sibling of top-p: keep the k most likely tokens instead of a probability mass. The two are often combined.

In [3]:
def top_k(probs, tokens, k=3):
    order = np.argsort(probs)[::-1][:k]
    return [tokens[i] for i in order]

print("top-k (k=3):", top_k(p, tokens, k=3))

top-k (k=3): ['mat', 'rug', 'sofa']


## Sample a next token

With the nucleus chosen, renormalize its probabilities and draw one token. A fixed seed makes the draw repeatable.

In [4]:
rng = np.random.default_rng(0)
idx = [tokens.index(t) for t in keep]
sub = p[idx] / p[idx].sum()
choice = rng.choice(keep, p=sub)
print("sampled token:", choice)

sampled token: mat


## Generate with a local model (Ollama)

Run a real model locally and sweep temperature and top-p. This needs Ollama running (`ollama pull llama3.2`); if it is not reachable, the cell prints a note and the rest of the notebook still works.

In [5]:
import requests

def generate(prompt, temperature, top_p, model="llama3.2"):
    r = requests.post("http://localhost:11434/api/generate", json={
        "model": model, "prompt": prompt, "stream": False,
        "options": {"temperature": temperature, "top_p": top_p},
    }, timeout=60)
    return r.json()["response"]

prompt = "In one sentence, describe a sunrise over the sea."
try:
    for T in [0.2, 1.0, 2.0]:
        print(f"--- temperature {T} ---")
        print(generate(prompt, temperature=T, top_p=0.9))
except Exception as e:
    print("Ollama not reachable here; run this locally with Ollama started.")
    print("(", type(e).__name__, ")")

--- temperature 0.2 ---
As the night's darkness slowly recedes, a vibrant tapestry of pinks, oranges, and purples unfurls across the horizon, gradually giving way to a brilliant blaze of golden light that dances across the waves as morning breaks over the sea.
--- temperature 1.0 ---
As the horizon slowly unfurls, a kaleidoscope of hues paints the sky with vibrant oranges, pinks, and purples, culminating in a radiant blaze of golden light that gently touches the rippling waves of the ocean.
--- temperature 2.0 ---
As the night's veil lifted, the horizon transformed into a kaleidoscope of warm hues - gentle pinks and soft oranges blending with wispy purples, softly seeping across the serene expanse of the sea, signaling the emergence of a new day amidst gentle ripples and peaceful waves.


## Your turn

**Micro-assignment.** Six problems on decoding; see `../micro-assignment/README.md`.

**Module 3 milestone.** Solve a real text task with a pretrained transformer and note how tokenization and attention shaped the result. Details in the milestone folder.

**Next, Module 4 (LLMs and GenAI):** build with these models, starting from prompting and structured outputs.